# 📖 Ahkeelak (أحكيلك) — Backend (FastAPI + ngrok + LangGraph)

*"Ahkeelak" is colloquial Arabic for "I'll tell you [a story]" — this loads
**Qwen2.5-7B-Instruct** and runs your PDF → RAG → **LangGraph** → story pipeline,
served over HTTP so the separate `app.py` (Streamlit, running on your own machine)
can call it with `requests`.

Works on **Google Colab** or **Kaggle Notebooks** — both give you a free GPU and
outbound internet access, which is all this notebook needs.

**How the language handling works:**
- **English** → the graph skips the translator node entirely; you get the story back as soon as it's written.
- **Arabic** → nothing is streamed until the Arabic translation is fully ready; the English draft is generated internally (translation needs it) but never sent to the client.
- **Bilingual** → the English story streams back as soon as the `writer` node finishes; translation keeps running in the background; the combined English + Arabic result streams once `translator`/`formatter` finish. English is never taken away once shown.

**Story length** (Short/Medium/Long) maps to its own `max_new_tokens` budget for both
the writer and the translator — no more than one length quietly sharing another's budget.

**Arabic quality guardrail:** Qwen2.5 occasionally code-switches into Chinese/Japanese/Korean
mid-translation. Each translated section is scored by what fraction of its letters are
actually Arabic script; low-scoring sections are cleaned and regenerated once with a
stronger instruction before falling back to the best attempt.

**How to use:**

*On Colab:*
1. Runtime → Change runtime type → **GPU** (T4 is enough for 4-bit).
2. Get a free ngrok authtoken: https://dashboard.ngrok.com/get-started/your-authtoken
3. Run every cell top to bottom. The last-but-one cell prints your public URL,
   e.g. `https://xxxx.ngrok-free.app`.
4. Paste that URL into `app.py`'s sidebar ("Backend URL") and run the Streamlit app
   on your own machine.

*On Kaggle:*
1. Settings (right panel) → Accelerator → **GPU T4 x2** (or P100) → Internet **On**.
2. Same ngrok authtoken as above.
3. Run every cell top to bottom, same as Colab.

Keep the notebook running while you use the Streamlit app — stopping it kills the
model and the server.


## 1. Install dependencies

In [1]:
!pip install -q fastapi "uvicorn[standard]" pyngrok nest_asyncio python-multipart \
    langchain langchain-classic langchain-chroma langchain-core langchain-huggingface \
    langchain-text-splitters langgraph transformers accelerate bitsandbytes \
    pymupdf sentence-transformers pydantic


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6

## 2. Imports

In [2]:
import os

# Must be set before torch initializes CUDA, or it has no effect. Reduces
# allocator fragmentation, which is what turns "7GB free" into failed
# 10GB allocations.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import re
import json
import gc
import uuid
import warnings
import threading
import tempfile
from pathlib import Path
from typing import List, Any, TypedDict

import fitz  # PyMuPDF
import torch
import nest_asyncio
import uvicorn
from pyngrok import ngrok, conf
from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.responses import StreamingResponse

from langgraph.graph import StateGraph
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import PydanticOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

warnings.filterwarnings("ignore")


## 3. Check GPU

In [3]:
print(f"Is CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("WARNING: no GPU detected. Runtime > Change runtime type > GPU, then re-run.")

print(f"Using device: {device}")


Is CUDA Available: True
GPU: Tesla T4
Using device: cuda


## 4. Config

In [4]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

# Caps that keep the prompt fed to the model bounded, which is what actually
# avoids CUDA OOM on 16GB GPUs (T4) — a long, unbounded RAG context is the
# most common cause of an out-of-memory error during the analyzer step.
RETRIEVER_K = 6            # chunks pulled per retrieval query
MAX_CONTEXT_CHARS = 12000  # hard cap on the context text sent to the analyzer


## 5. Load the model + embeddings

In [5]:
## Initialize Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    # sdpa uses a fused attention kernel instead of materializing the full
    # (seq_len x seq_len) attention matrix — this is the single biggest lever
    # against CUDA OOM on long RAG contexts on a 16GB GPU.
    attn_implementation="sdpa",
)
model.eval()

print("Model loaded successfully.")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully.


In [7]:
## Load the embedding model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 6. LLM service wrapper

In [8]:
class LLMService:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    @torch.inference_mode()
    def generate(self, system_prompt: str, user_prompt: str, max_new_tokens: int = 1200) -> str:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        prompt = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=self.tokenizer.eos_token_id,
            use_cache=True,
        )
        generated = outputs[0][inputs["input_ids"].shape[-1]:]
        text = self.tokenizer.decode(generated, skip_special_tokens=True).strip()

        # Release cached (but unused) GPU memory now rather than letting it
        # accumulate as fragmentation across the many generate() calls in a
        # single request (analyzer -> writer -> one call per translated section).
        del inputs, outputs, generated
        gc.collect()
        torch.cuda.empty_cache()

        text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
        text = re.sub(r"<thinking>.*?</thinking>", "", text, flags=re.DOTALL | re.IGNORECASE)
        text = re.sub(r"Okay, let me .*?(?=\n\n|\Z)", "", text, flags=re.DOTALL | re.IGNORECASE)
        return text.strip()


LLM = LLMService(model, tokenizer)


## 7. PDF processing

In [9]:
class PDFProcessor:
    def extract(self, pdf_path):
        doc = fitz.open(pdf_path)
        pages, full_text = [], []
        for page_num, page in enumerate(doc, start=1):
            text = page.get_text("text")
            pages.append({"page": page_num, "text": text})
            full_text.append(text)
        doc.close()
        return {"pages": pages, "full_text": "\n".join(full_text)}


## 8. Prompts + length/token budgets

`LENGTH_GUIDE` is the *word-count instruction* shown to the model.
`LENGTH_TOKENS` / `TRANSLATE_TOKENS` are the actual `max_new_tokens` budgets — each
length tier gets its own budget for both the writer and the translator.

In [10]:
RESEARCH_ANALYZER_PROMPT = """
You are an expert scientific research analyst.

Your task is to analyze ONLY the provided research paper context.

ABSOLUTE RULES:
- Do NOT invent information.
- Do NOT infer facts not explicitly supported by the context.
- Do NOT invent researcher names, universities, laboratories, cities, companies, datasets, experiments, numerical results, or citations.
- Use ONLY the provided context.
- If something is not in the context, write exactly "Not specified".
- Stay completely faithful to the paper.
- Whenever possible, remember which page contains important information.

{format_instructions}

Research Context:

{context}
"""

WRITER_PROMPT = """
You are an award-winning scientific storyteller.

Your task is to transform the research analysis into an engaging story.

IMPORTANT:
- Write ONLY in English.
- Do NOT translate.
- Do NOT include Arabic.
- Do NOT add titles like "# English" or "# Arabic".
- Return ONLY the story.
- Target length: {story_length_words}. Stay within this range.

Rules:
- NEVER invent facts, researcher names, universities, laboratories, cities, companies, datasets, experiments, numerical results, or events not supported by the analysis.

If the paper does not mention names, use: The researchers / The research team / The study / The authors.

Creativity should come ONLY from storytelling.

Story Style:
{story_style}

Target Audience:
{audience}

Research Title:
{title}

Research Problem:
{problem}

Motivation:
{motivation}

Methodology:
{methodology}

Datasets:
{datasets}

Experiments:
{experiments}

Key Findings:
{findings}

Limitations:
{limitations}

Future Work:
{future_work}

Main Contribution:
{contribution}
"""

## ===========================================================================
## ============================== Guides =====================================

STYLE_GUIDE = {
    "Detective": """
Structure the story as a detective investigation.
Imagine you are explaining a scientific investigation.
Make it exciting and grab the attention.

Structure:
1. The Mystery
2. The First Clues
3. Following the Evidence
4. The Investigation
5. The Breakthrough
6. Solving the Case
7. Lessons Learned

Interpretation:
- The mystery is the research problem.
- The clues are observations and experiments.
- The evidence is the reported results.
- The culprit is the root cause or challenge identified by the paper.
- The solution is the paper's contribution.

Never invent detectives, police, criminals, laboratories, cities, or universities.
The researchers remain "the research team" unless names explicitly appear in the paper.
""",
    "Adventure": """
Write the research as a scientific expedition.

Structure:
1. The Mission
2. Preparing for the Journey
3. Entering Unknown Territory
4. Obstacles Encountered
5. Important Discoveries
6. Reaching the Destination
7. What Comes Next

Do not invent islands, jungles, ships, or fictional locations.
The adventure comes from scientific discovery.
""",
    "Sci-Fi": """
Present the research as if explaining an advanced technology from the near future.

Structure:
1. Future Challenge
2. Existing Technology
3. New Innovation
4. How It Works
5. Real-World Impact
6. Future Possibilities

Do NOT invent future inventions, companies, or space travel.
The futuristic feeling must come ONLY from the narration, while every scientific fact remains unchanged.
""",
    "Fantasy": """
Retell the research as an ancient legend.

Structure:
1. The Ancient Challenge
2. The Lost Knowledge
3. The Trials
4. The Hidden Wisdom
5. The Final Revelation
6. The Legacy

Never invent magical creatures, kingdoms, spells, or fictional characters.
The fantasy atmosphere must come only from the narration.
""",
}

AUDIENCE_GUIDE = {
    "Children": "Use simple vocabulary, short sentences, friendly explanations, and avoid jargon.",
    "High School": "Explain concepts clearly, introduce technical terms with brief definitions, and use relatable examples.",
    "University": "Maintain technical accuracy while keeping the narrative engaging. Assume readers have basic scientific knowledge.",
    "Researchers": "Preserve scientific terminology and methodology. Avoid oversimplification and focus on precision.",
}

LENGTH_GUIDE = {
    "Short": "400-600 words",
    "Medium": "700-900 words",
    "Long": "1000-1300 words",
}

# max_new_tokens for the story itself, one budget per length tier
LENGTH_TOKENS = {
    "Short": 850,
    "Medium": 1300,
    "Long": 1900,
}

# max_new_tokens per translated *section* (translation runs section-by-section)
TRANSLATE_TOKENS = {
    "Short": 500,
    "Medium": 650,
    "Long": 850,
}




## 9. Structured output parser + retrieval queries

In [11]:
class ResearchAnalysis(BaseModel):
    title: str = Field(description="Exact title of the paper if present, otherwise 'Not specified'")
    research_problem: str = Field(description="The core research problem the paper addresses")
    motivation: str = Field(description="Why this problem matters / the gap being filled")
    methodology: str = Field(description="Detailed description of the proposed method / algorithm")
    datasets: List[str] = Field(description="List of datasets used. Empty list if none mentioned")
    experiments: List[str] = Field(description="List of key experiments / evaluation setups")
    key_findings: List[str] = Field(description="List of main experimental results and findings")
    limitations: str = Field(description="Limitations discussed in the paper")
    future_work: str = Field(description="Future work or open directions mentioned")
    main_contribution: str = Field(description="The primary contribution of the paper")


parser = PydanticOutputParser(pydantic_object=ResearchAnalysis)


In [12]:
RETRIEVAL_QUERIES = [
    "What is the title, abstract, and main research problem of this paper?",
    "What is the motivation and the gap this paper addresses?",
    "Describe the proposed methodology, algorithm, and key technical ideas in detail.",
    "What datasets, experimental setup, baselines, and evaluation metrics were used?",
    "What are the main experimental results, key findings, and quantitative outcomes?",
    "What limitations, ablation studies, and future work does the paper discuss?",
    "What is the main contribution and how does it differ from previous work?",
]

## 10. Arabic-quality guardrail

Qwen2.5 occasionally code-switches into Chinese/Japanese/Korean mid-translation.
`clean_arabic_text` strips those characters out; `arabic_ratio` measures what
fraction of the letters in a section are actually Arabic script, so the translator
node can detect a bad generation and retry it once with a stronger instruction.

In [13]:
FOREIGN_SCRIPT = re.compile(r"[\u4e00-\u9fff\u3040-\u30ff\u3400-\u4dbf\uac00-\ud7af\u0e00-\u0e7f]+")
ARABIC_CHAR = re.compile(r"[\u0600-\u06FF]")


def clean_arabic_text(text: str) -> str:
    text = FOREIGN_SCRIPT.sub("", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def arabic_ratio(text: str) -> float:
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    arabic = sum(1 for c in letters if ARABIC_CHAR.match(c))
    return arabic / len(letters)


ARABIC_TRANSLATOR_SYSTEM = """أنت مترجم علمي عربي متخصص. أخرج الترجمة بالغة العربية الفصحى فقط.

قواعد صارمة:
- ممنوع منعاً باتاً كتابة أي حرف صيني أو ياباني أو كوري مهما كان السبب.
- ممنوع كتابة أي جملة كاملة بالإنجليزية؛ يُسمح فقط بالمصطلحات التقنية الشائعة (مثل Transformer، RAG، LLM، GPU) داخل جملة عربية.
- أخرج فقط النص العربي المترجم دون أي تعليقات أو مقدمات أو رموز Markdown إضافية.
- حافظ على كل حقيقة علمية ورقم واسم كما هو.
"""

# ARABIC_TRANSLATOR_SYSTEM = """\
# \u0623\u0646\u062a \u0645\u062a\u0631\u062c\u0645 \u0639\u0644\u0645\u064a \u0639\u0631\u0628\u064a \u0645\u062a\u062e\u0635\u0635. \u0623\u062e\u0631\u062c \u0627\u0644\u062a\u0631\u062c\u0645\u0629 \u0628\u0627\u0644\u0644\u063a\u0629 \u0627\u0644\u0639\u0631\u0628\u064a\u0629 \u0627\u0644\u0641\u0635\u062d\u0649 \u0641\u0642\u0637.

# \u0642\u0648\u0627\u0639\u062f \u0635\u0627\u0631\u0645\u0629:
# - \u0645\u0645\u0646\u0648\u0639 \u0645\u0646\u0639\u0627\u064b \u0628\u0627\u062a\u0627\u064b \u0643\u062a\u0627\u0628\u0629 \u0623\u064a \u062d\u0631\u0641 \u0635\u064a\u0646\u064a \u0623\u0648 \u064a\u0627\u0628\u0627\u0646\u064a \u0623\u0648 \u0643\u0648\u0631\u064a \u0645\u0647\u0645\u0627 \u0643\u0627\u0646 \u0627\u0644\u0633\u0628\u0628.
# - \u0645\u0645\u0646\u0648\u0639 \u0643\u062a\u0627\u0628\u0629 \u0623\u064a \u062c\u0645\u0644\u0629 \u0643\u0627\u0645\u0644\u0629 \u0628\u0627\u0644\u0625\u0646\u062c\u0644\u064a\u0632\u064a\u0629\u061b \u064a\u064f\u0633\u0645\u062d \u0641\u0642\u0637 \u0628\u0627\u0644\u0645\u0635\u0637\u0644\u062d\u0627\u062a \u0627\u0644\u062a\u0642\u0646\u064a\u0629 \u0627\u0644\u0634\u0627\u0626\u0639\u0629 (\u0645\u062b\u0644 Transformer\u060c RAG\u060c LLM\u060c GPU) \u062f\u0627\u062e\u0644 \u062c\u0645\u0644\u0629 \u0639\u0631\u0628\u064a\u0629.
# - \u0623\u062e\u0631\u062c \u0641\u0642\u0637 \u0627\u0644\u0646\u0635 \u0627\u0644\u0639\u0631\u0628\u064a \u0627\u0644\u0645\u062a\u0631\u062c\u0645 \u062f\u0648\u0646 \u0623\u064a \u062a\u0639\u0644\u064a\u0642\u0627\u062a \u0623\u0648 \u0645\u0642\u062f\u0645\u0627\u062a \u0623\u0648 \u0631\u0645\u0648\u0632 Markdown \u0625\u0636\u0627\u0641\u064a\u0629.
# - \u062d\u0627\u0641\u0638 \u0639\u0644\u0649 \u0643\u0644 \u062d\u0642\u064a\u0642\u0629 \u0639\u0644\u0645\u064a\u0629 \u0648\u0631\u0642\u0645 \u0648\u0627\u0633\u0645 \u0643\u0645\u0627 \u0647\u0648.
# """


## 11. LangGraph pipeline

A real `StateGraph`: retrieve → context → analyze → write → *(conditionally)* translate → format.
The conditional edge (`should_translate`) skips the translator node entirely when
`output_language == "English"`.

In [14]:
def split_story_into_sections(story: str):
    parts = re.split(r"(?=^#{1,3}\s+.+$)", story, flags=re.MULTILINE)
    sections = []
    for part in parts:
        part = part.strip()
        if not part:
            continue
        lines = part.split("\n", 1)
        heading = lines[0].strip()
        content = lines[1].strip() if len(lines) > 1 else ""
        sections.append((heading, content))
    return sections

In [15]:
class StoryState(TypedDict):
    ## Per-request inputs
    story_style: str
    audience: str
    story_length: str
    output_language: str
    retriever: Any

    ## RAG
    retrieved: List[Document]
    context: str
    citations: List[int]

    ## AI Pipeline
    analysis: dict
    english_story: str
    arabic_story: str
    final_story: str

In [16]:
def retrieve_node(state: StoryState):
    retriever = state["retriever"]
    docs, seen = [], set()
    for query in RETRIEVAL_QUERIES:
        for doc in retriever.invoke(query):
            key = (doc.metadata["page"], doc.page_content[:100])
            if key not in seen:
                seen.add(key)
                docs.append(doc)
    return {"retrieved": docs}


def context_node(state: StoryState):
    docs = sorted(state["retrieved"], key=lambda d: d.metadata["page"])
    sections, citations, total_chars = [], [], 0
    for doc in docs:
        if total_chars >= MAX_CONTEXT_CHARS:
            break
        page = doc.metadata["page"]
        citations.append(page)
        section = f"""
=====================================
Page {page}
=====================================

        {doc.page_content}
        """
        sections.append(section)
        total_chars += len(section)
    return {"context": "\n\n".join(sections), "citations": sorted(set(citations))}


def analyzer_node(state: StoryState):
    format_instructions = parser.get_format_instructions()
    prompt = RESEARCH_ANALYZER_PROMPT.format(context=state["context"], format_instructions=format_instructions)
    response = LLM.generate(
        system_prompt="You are an expert research analyst. Always return valid structured output exactly as requested. Never invent facts.",
        user_prompt=prompt,
        max_new_tokens=1500,
    )
    try:
        analysis_dict = parser.parse(response).model_dump()
    except Exception as e:
        print(f"Pydantic parsing failed: {e}")
        analysis_dict = {
            "title": "Not specified", "research_problem": "Not specified", "motivation": "Not specified",
            "methodology": "Not specified", "datasets": [], "experiments": [], "key_findings": [],
            "limitations": "Not specified", "future_work": "Not specified", "main_contribution": "Not specified",
        }
    return {"analysis": analysis_dict}


def writer_node(state: StoryState):
    analysis = state["analysis"]
    story_length = state["story_length"]
    prompt = WRITER_PROMPT.format(
        story_style=STYLE_GUIDE[state["story_style"]],
        audience=AUDIENCE_GUIDE[state["audience"]],
        story_length_words=LENGTH_GUIDE[story_length],
        title=analysis["title"], problem=analysis["research_problem"], motivation=analysis["motivation"],
        methodology=analysis["methodology"], datasets=", ".join(analysis["datasets"]),
        experiments=", ".join(analysis["experiments"]),
        findings="\n".join("- " + x for x in analysis["key_findings"]),
        limitations=analysis["limitations"], future_work=analysis["future_work"],
        contribution=analysis["main_contribution"],
    )
    story = LLM.generate(
        system_prompt="""You are a scientific storytelling assistant.

CRITICAL RULES (MUST FOLLOW):
1. Only use information that appears in the Research Analysis provided below.
2. If a field is "Not specified" or empty, omit that topic completely.
3. NEVER invent scientific content, mechanisms, experiments, datasets, numbers, or results.
4. Output ONLY the final story. No reasoning, planning, or <think> blocks.
5. Start directly with the first section heading.
6. Respect the requested target length.
7. The story must be about the EXACT research described in the analysis.

If the analysis is empty or does not describe a real paper, reply with exactly:
"ERROR: Insufficient research analysis provided. Cannot generate story."

Always write ONLY in English.""",
        user_prompt=prompt,
        max_new_tokens=LENGTH_TOKENS[story_length],
    )
    return {"english_story": story}


def should_translate(state: StoryState):
    return "translator" if state["output_language"] in ("Arabic", "Bilingual") else "formatter"


def translator_node(state: StoryState):
    english_story = state["english_story"]
    section_tokens = TRANSLATE_TOKENS[state["story_length"]]
    sections = split_story_into_sections(english_story)
    arabic_sections = []

    for heading, content in sections:
        if not content.strip():
            arabic_sections.append(heading)
            continue

        def build_prompt(extra_warning=""):
            return f"""
Translate the following section of a scientific story into fluent Modern Standard Arabic.

{extra_warning}
Rules:
- Keep the heading structure (translate the heading too).
- Preserve every scientific fact, number, and name exactly.
- Do NOT summarize or add commentary.
- Output ONLY the Arabic translation — no Chinese, no Japanese, no Korean, no English
  sentences (technical terms like Transformer/RAG/LLM/GPU may stay in English inside
  an Arabic sentence).

English section:
{heading}

{content}
"""

        arabic_part = LLM.generate(
            system_prompt=ARABIC_TRANSLATOR_SYSTEM,
            user_prompt=build_prompt(),
            max_new_tokens=section_tokens,
        )
        arabic_part = clean_arabic_text(arabic_part)

        # Too little of the output is actually Arabic script -> likely code-switched
        # into Chinese/English. Retry once with a stronger warning.
        if arabic_ratio(arabic_part) < 0.6:
            retry = LLM.generate(
                system_prompt=ARABIC_TRANSLATOR_SYSTEM,
                user_prompt=build_prompt(
                    extra_warning="تحذير: المحاولة السابقة حادت عن العربية. اكتب بالعربية الفصحى فقط ولا تكتب أي حرف صيني.\n"
                ),
                max_new_tokens=section_tokens,
            )
            retry = clean_arabic_text(retry)
            if arabic_ratio(retry) > arabic_ratio(arabic_part):
                arabic_part = retry

        arabic_sections.append(arabic_part)

    return {"arabic_story": "\n\n".join(arabic_sections)}


def formatter_node(state: StoryState):
    output_language = state["output_language"]
    english_story = state.get("english_story", "")
    arabic_story = state.get("arabic_story", "")
    citations = state.get("citations", [])

    if output_language == "English":
        output = english_story
    elif output_language == "Arabic":
        output = arabic_story
    else:
        output = f"""
# English

{english_story}

---

# العربية

{arabic_story}
"""
    output += f"""

---
## 📚 Source Pages

{','.join(map(str, citations))}
"""
    return {"final_story": output}


In [17]:
builder = StateGraph(StoryState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("context", context_node)
builder.add_node("analyzer", analyzer_node)
builder.add_node("writer", writer_node)
builder.add_node("translator", translator_node)
builder.add_node("formatter", formatter_node)

builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "context")
builder.add_edge("context", "analyzer")
builder.add_edge("analyzer", "writer")
builder.add_conditional_edges("writer", should_translate, {"translator": "translator", "formatter": "formatter"})
builder.add_edge("translator", "formatter")
builder.set_finish_point("formatter")

graph = builder.compile()
print("LangGraph compiled:", list(graph.get_graph().nodes))

LangGraph compiled: ['__start__', 'retrieve', 'context', 'analyzer', 'writer', 'translator', 'formatter', '__end__']


## 12. Streaming pipeline runner

In [18]:
def stream_pipeline(pdf_path: str, story_style: str, audience: str, story_length: str, output_language: str):
    processor = PDFProcessor()
    pdf = processor.extract(pdf_path)

    docs = [
        Document(page_content=p["text"], metadata={"page": p["page"], "source": pdf_path, "chunk_type": "text"})
        for p in pdf["pages"]
    ]
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks = splitter.split_documents(docs)

    vector_db = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        collection_name=f"paper_{uuid.uuid4().hex}",
    )
    retriever = vector_db.as_retriever(search_kwargs={"k": RETRIEVER_K}, search_type="similarity")

    initial_state = {
        "story_style": story_style,
        "audience": audience,
        "story_length": story_length,
        "output_language": output_language,
        "retriever": retriever,
    }

    sent_english = False
    for state in graph.stream(initial_state, stream_mode="values"):
        # Only preview English for "English" or "Bilingual". For Arabic-only, stay
        # silent until the final Arabic text is ready — never flash English at
        # an Arabic-only user.
        if not sent_english and state.get("english_story") and output_language != "Arabic":
            sent_english = True
            yield json.dumps({
                "stage": "english",
                "english_story": state["english_story"],
                "analysis": state.get("analysis", {}),
                "citations": state.get("citations", []),
            }) + "\n"

        if state.get("final_story"):
            yield json.dumps({
                "stage": "final",
                "final_story": state["final_story"],
                "english_story": state.get("english_story", ""),
                "arabic_story": state.get("arabic_story", ""),
                "analysis": state.get("analysis", {}),
                "citations": state.get("citations", []),
            }) + "\n"


## 13. FastAPI app (streaming NDJSON response)

In [19]:
app = FastAPI(title="Ahkeelak API", description="Turn a research paper into a story, in English and/or Arabic.")

VALID_STYLES = list(STYLE_GUIDE.keys())
VALID_AUDIENCES = list(AUDIENCE_GUIDE.keys())
VALID_LENGTHS = list(LENGTH_GUIDE.keys())
VALID_LANGUAGES = ["English", "Arabic", "Bilingual"]


@app.get("/health")
def health():
    return {
        "app": "Ahkeelak",
        "status": "ok",
        "model": MODEL_NAME,
        "device": str(device),
        "styles": VALID_STYLES,
        "audiences": VALID_AUDIENCES,
        "lengths": VALID_LENGTHS,
        "languages": VALID_LANGUAGES,
    }


@app.post("/generate-story")
async def generate_story(
    file: UploadFile = File(...),
    story_style: str = Form("Detective"),
    audience: str = Form("University"),
    story_length: str = Form("Medium"),
    output_language: str = Form("English"),
):
    if story_style not in VALID_STYLES:
        raise HTTPException(400, f"story_style must be one of {VALID_STYLES}")
    if audience not in VALID_AUDIENCES:
        raise HTTPException(400, f"audience must be one of {VALID_AUDIENCES}")
    if story_length not in VALID_LENGTHS:
        raise HTTPException(400, f"story_length must be one of {VALID_LENGTHS}")
    if output_language not in VALID_LANGUAGES:
        raise HTTPException(400, f"output_language must be one of {VALID_LANGUAGES}")
    if not file.filename.lower().endswith(".pdf"):
        raise HTTPException(400, "Only PDF files are supported")

    tmp_dir = Path(tempfile.gettempdir()) / "ahkeelak_uploads"
    tmp_dir.mkdir(parents=True, exist_ok=True)
    tmp_path = tmp_dir / f"{uuid.uuid4().hex}_{file.filename}"
    with open(tmp_path, "wb") as f:
        f.write(await file.read())

    def event_generator():
        try:
            for chunk in stream_pipeline(str(tmp_path), story_style, audience, story_length, output_language):
                yield chunk
        except Exception as e:
            yield json.dumps({"stage": "error", "detail": str(e)}) + "\n"
        finally:
            if tmp_path.exists():
                tmp_path.unlink()

    return StreamingResponse(event_generator(), media_type="application/x-ndjson")


## 14. Start ngrok tunnel + server

In [20]:
NGROK_AUTHTOKEN = "3EcpTBJLJGe5P8lk1DnMWaq9UhD_6ofZTVGSjamhbbenHdKRw"

conf.get_default().auth_token = NGROK_AUTHTOKEN

# Close any tunnels left open from a previous run in this session
ngrok.kill()

public_url = ngrok.connect(8000, "http")
print(f"\nPublic backend URL: {public_url.public_url}")
print("Paste this into streamlit_app.py's sidebar (Backend URL field).\n")

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("Server starting... give it a few seconds, then test with:")
print(f"  curl {public_url.public_url}/health")



Public backend URL: https://sardine-spinster-winnings.ngrok-free.dev
Paste this into streamlit_app.py's sidebar (Backend URL field).

Server starting... give it a few seconds, then test with:
  curl https://sardine-spinster-winnings.ngrok-free.dev/health


## 15. (Optional) quick test from inside the notebook

In [21]:
import requests
import time

time.sleep(3)
try:
    r = requests.get(f"{public_url.public_url}/health", timeout=10)
    print(r.status_code, r.json())
except Exception as e:
    print("Not up yet, wait a few seconds and re-run this cell:", e)


INFO:     Started server process [907]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     34.125.1.84:0 - "GET /health HTTP/1.1" 200 OK
200 {'app': 'Ahkeelak', 'status': 'ok', 'model': 'Qwen/Qwen2.5-7B-Instruct', 'device': 'cuda', 'styles': ['Detective', 'Adventure', 'Sci-Fi', 'Fantasy'], 'audiences': ['Children', 'High School', 'University', 'Researchers'], 'lengths': ['Short', 'Medium', 'Long'], 'languages': ['English', 'Arabic', 'Bilingual']}
